In [1]:
# =========================
# 0) Config + Imports
# =========================

import os
import re
import glob
import json
import numpy as np
import faiss
from typing import List, Dict, Tuple

from rank_bm25 import BM25Okapi
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer, CrossEncoder

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

Section 1 — Load documents + quick preview

In [2]:
# =========================
# 1) Load Documents
# =========================

DOC_GLOB = "/home/jupyter/advanced/nlp/documents/*.txt"  # adjust if needed

documents = []
doc_ids = []

for filepath in sorted(glob.glob(DOC_GLOB)):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    documents.append(text)
    doc_ids.append(os.path.basename(filepath).replace(".txt", ""))

print("Loaded documents:", len(documents))
print("First doc id:", doc_ids[0])
print("\nFirst doc preview:\n")
print(documents[0][:600])

Loaded documents: 296
First doc id: DOC-0001

First doc preview:

# The Dissolution of Wampa Robotics: A Structural Analysis of Coordinated Corporate Action Within the CGC Framework

**Haven Institute for Policy Studies | Research Paper**
**Dr. Elara Quinshan**
**74-03-12 PCE**
Classification: OPEN

---

## Abstract

The coordinated destruction of Wampa Robotics in 28 PCE remains one of the most consequential acts of collective megacorporate force in the Post-Cascade Era. Executed by the combined forces of the Original 5 Firms at a ceremony commemorating the 25th anniversary of the Clairos Governance Council, the event inaugurated a new phase of inter-megaco


Section 2 — Chunking (keep original text)

In [3]:
# =========================
# 2) Chunking
# =========================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
chunk_to_doc = []

for d_id, doc in zip(doc_ids, documents):
    split_chunks = text_splitter.split_text(doc)
    for ch in split_chunks:
        chunks.append(ch)
        chunk_to_doc.append(d_id)

print("Total chunks:", len(chunks))
print("\nSample chunk:\n")
print("DOC:", chunk_to_doc[0])
print(chunks[0][:500])

Total chunks: 4074

Sample chunk:

DOC: DOC-0001
# The Dissolution of Wampa Robotics: A Structural Analysis of Coordinated Corporate Action Within the CGC Framework

**Haven Institute for Policy Studies | Research Paper**
**Dr. Elara Quinshan**
**74-03-12 PCE**
Classification: OPEN

---

## Abstract


Section 3 — Build embeddings + FAISS index

In [4]:
# =========================
# 3) Dense embeddings + FAISS
# =========================

embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

chunk_embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(chunk_embeddings)

print("FAISS ready:", faiss_index.ntotal, "vectors")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/128 [00:00<?, ?it/s]

FAISS ready: 4074 vectors


Section 4 — BM25 tokenization (don’t just .split())

In [5]:
# =========================
# 4) BM25 (better tokenization)
# =========================

_token_re = re.compile(r"[a-zA-Z0-9]+")

def bm25_tokenize(text: str) -> List[str]:
    # Extract alphanumerics; splits on hyphens/punct automatically.
    # Keeps numbers like 77, 031, WH, 2, DOC, 0255, etc.
    return _token_re.findall(text.lower())

tokenized_chunks = [bm25_tokenize(ch) for ch in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 ready")

BM25 ready


Section 5 — Hybrid retrieval (dense + sparse candidate pool)

In [6]:
# =========================
# 5) Hybrid Retrieval
# =========================

def hybrid_retrieve(
    question: str,
    dense_k: int = 20,
    sparse_k: int = 20,
    top_k: int = 8,
    alpha: float = 0.65
) -> List[Dict]:
    # Dense
    q_emb = embedding_model.encode([question], normalize_embeddings=True).astype("float32")
    D, I = faiss_index.search(q_emb, k=dense_k)

    dense_scores = {int(idx): float(score) for score, idx in zip(D[0], I[0]) if idx >= 0}

    # Sparse
    q_tok = bm25_tokenize(question)
    bm25_scores = bm25.get_scores(q_tok)

    # top sparse indices
    if sparse_k >= len(bm25_scores):
        top_sparse_idx = np.argsort(bm25_scores)[::-1]
    else:
        top_sparse_idx = np.argpartition(bm25_scores, -sparse_k)[-sparse_k:]

    sparse_scores = {int(i): float(bm25_scores[i]) for i in top_sparse_idx}

    candidates = sorted(set(dense_scores.keys()) | set(sparse_scores.keys()))

    # normalize sparse on candidates
    svals = np.array([sparse_scores.get(i, 0.0) for i in candidates], dtype="float32")
    smin, smax = float(svals.min()), float(svals.max())
    def ns(v): return (v - smin) / (smax - smin + 1e-9)

    combined = []
    for idx in candidates:
        d = dense_scores.get(idx, 0.0)
        s = ns(sparse_scores.get(idx, 0.0))
        combined.append((idx, alpha * d + (1 - alpha) * s))

    combined.sort(key=lambda x: x[1], reverse=True)
    combined = combined[:top_k]

    results = []
    for idx, score in combined:
        results.append({
            "chunk_index": idx,
            "doc_id": chunk_to_doc[idx],
            "chunk": chunks[idx],
            "score": float(score),
        })
    return results

Section 6 — Reranker (CrossEncoder) + dedup docs

In [7]:
# =========================
# 6) Reranking
# =========================

reranker = CrossEncoder("BAAI/bge-reranker-base")

def rerank(question: str, retrieved: List[Dict], top_k: int = 8) -> List[Dict]:
    pairs = [[question, r["chunk"]] for r in retrieved]
    scores = reranker.predict(pairs)

    out = []
    for r, s in zip(retrieved, scores):
        out.append({**r, "rerank_score": float(s)})

    out.sort(key=lambda x: x["rerank_score"], reverse=True)
    return out[:top_k]

def dedup_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [8]:
from typing import Optional

def retrieve_ranked_doc_ids(question: str, top_k_docs: int = 50) -> List[str]:
    """
    Returns a ranked list of doc_ids based on chunk reranking.
    We'll convert chunk ranking -> doc ranking by taking the best (highest) chunk score per doc.
    """
    retrieved = hybrid_retrieve(question, top_k=200)          # large candidate pool
    reranked = rerank(question, retrieved, top_k=200)         # rerank lots of chunks

    # Aggregate chunk-level rerank scores into doc-level scores (max score per doc)
    best_score_by_doc = {}
    for r in reranked:
        d = r["doc_id"]
        s = r["rerank_score"]
        if d not in best_score_by_doc or s > best_score_by_doc[d]:
            best_score_by_doc[d] = s

    # Sort docs by best score desc
    ranked_docs = sorted(best_score_by_doc.items(), key=lambda x: x[1], reverse=True)
    ranked_doc_ids = [d for d, _ in ranked_docs][:top_k_docs]
    return ranked_doc_ids

Section 7 — L4/L5 detection (“abstain” logic)

If top evidence is weak OR the LLM says NOT_FOUND → return empty answer.
But whether docs should be empty (L4) or non-empty (L5) is tricky.
You can’t truly know L4 vs L5, but you can do a good heuristic:
If retrieval is “confident” that the question relates to corpus but the answer isn’t supported → treat like L5 (return docs but empty answer).
If retrieval confidence is low / query looks real-world (FDA, Microsoft market cap, exchange rate, Paris Agreement) → treat like L4 (empty docs, empty answer).

In [9]:
# =========================
# 7) Unanswerable (L4/L5) detection heuristics
# =========================

REAL_WORLD_HINTS = [
    "2024", "fda", "microsoft", "samsung", "spacex", "falcon 9",
    "paris agreement", "euro", "usd", "exchange rate", "market cap",
    "international maritime organization", "imo", "dubai", "jebel ali",
    "european space agency", "esa"
]

def looks_real_world(question: str) -> bool:
    q = question.lower()
    return any(h in q for h in REAL_WORLD_HINTS)

def retrieval_confidence(reranked: List[Dict]) -> float:
    # CrossEncoder scores are typically in [0,1] for bge-reranker-base
    # Not guaranteed, but usually. We'll clamp just in case.
    if not reranked:
        return 0.0
    return float(max(0.0, min(1.0, reranked[0]["rerank_score"])))

def should_abstain(question: str, reranked: List[Dict], conf_threshold: float = 0.35) -> Tuple[bool, str]:
    """
    Returns (abstain, mode) where mode is:
      - "L4": empty answer + empty docs
      - "L5": empty answer + non-empty docs
      - "NO": produce answer
    """
    conf = retrieval_confidence(reranked)

    # If it looks like real-world, default to L4 unless retrieval is very confident it's in-corpus
    if looks_real_world(question) and conf < 0.55:
        return True, "L4"

    # If retrieval is weak, abstain L4 (likely not in corpus)
    if conf < conf_threshold:
        return True, "L4"

    # Otherwise: question seems in-corpus but may still be unanswerable / false premise
    # We decide after generation; this is a pre-gate only.
    return False, "NO"

Section 8 — LLM: prompt + safer generation + parsing

In [10]:
# =========================
# 8) LLM Answer Generator
# =========================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # keep small for speed
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

def build_prompt(question: str, contexts: List[str]) -> str:
    ctx = "\n\n".join([f"[PASSAGE {i+1}]\n{c}" for i, c in enumerate(contexts)])
    return f"""You are a strict retrieval QA system.

Rules:
- Use ONLY the passages below.
- If the passages do not contain enough information to answer, output exactly: NOT_FOUND
- Output ONLY the answer text, nothing else.
- Keep the answer concise (<= 20 words if possible).

Passages:
{ctx}

Question:
{question}

Answer:
"""

def llm_answer(prompt: str, max_new_tokens: int = 64) -> str:
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,   # important: only newly generated text
    )[0]["generated_text"]
    return out.strip()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Section 9 — Full pipeline: retrieve → rerank → generate → postprocess (L4/L5)

In [11]:
# =========================
# 9) Full RAG Pipeline w/ L4/L5 behavior
# =========================

def answer_question(question: str) -> Dict:
    retrieved = hybrid_retrieve(question, top_k=30)
    reranked = rerank(question, retrieved, top_k=10)

    # Pre-gate (cheap)
    abstain, mode = should_abstain(question, reranked, conf_threshold=0.35)
    if abstain and mode == "L4":
        return {"answer": "", "documents": []}

    # Use top N contexts (keep small for speed)
    top_contexts = [r["chunk"] for r in reranked[:3]]
    prompt = build_prompt(question, top_contexts)
    ans = llm_answer(prompt, max_new_tokens=64)

    # If model says NOT_FOUND => empty answer
    if ans.strip().upper().startswith("NOT_FOUND"):
        # Decide L4 vs L5 now:
        # If question looked real-world => L4
        # Else if retrieval confidence is decent => L5 (return docs)
        conf = retrieval_confidence(reranked)
        if looks_real_world(question) and conf < 0.65:
            return {"answer": "", "documents": []}
        else:
            doc_list = dedup_preserve_order([r["doc_id"] for r in reranked[:5]])
            return {"answer": "", "documents": doc_list}

    # Normal answer path
    doc_list = dedup_preserve_order([r["doc_id"] for r in reranked[:5]])

    # Light cleanup: remove surrounding quotes
    ans = ans.strip().strip('"').strip()

    return {"answer": ans, "documents": doc_list}

Section 10 — Quick tests (including L4/L5 examples from your jsonl)

In [12]:
# =========================
# 12) Retrieval evaluation metrics (MRR, Recall@K, Precision@K)
# =========================

from collections import defaultdict

def _reciprocal_rank(pred_docs: List[str], gold_docs: List[str], k: int) -> float:
    """RR@k: 1/rank of first relevant doc in top-k, else 0."""
    gold = set(gold_docs)
    for i, d in enumerate(pred_docs[:k], start=1):
        if d in gold:
            return 1.0 / i
    return 0.0

def _recall_at_k(pred_docs: List[str], gold_docs: List[str], k: int) -> float:
    """Recall@k for doc retrieval: did we hit any gold doc? (binary recall)."""
    gold = set(gold_docs)
    return 1.0 if any(d in gold for d in pred_docs[:k]) else 0.0

def _precision_at_k(pred_docs: List[str], gold_docs: List[str], k: int) -> float:
    """Precision@k for doc retrieval: (# relevant in top-k)/k.
    Note: gold set is usually small (often 1 doc), so precision will be low unless k small.
    """
    gold = set(gold_docs)
    top = pred_docs[:k]
    if not top:
        return 0.0
    rel = sum(1 for d in top if d in gold)
    return rel / float(k)

def evaluate_retrieval(instances: List[Dict], preds: List[Dict], ks=(1,3,5)) -> Dict:
    """
    instances: list of dataset rows (from jsonl)
    preds: list of {"answer": str, "documents": [doc_ids...]} aligned with instances
    """
    assert len(instances) == len(preds)

    # buckets
    by_diff = defaultdict(list)   # difficulty -> list of indices
    for i, ex in enumerate(instances):
        by_diff[ex.get("difficulty", "UNK")].append(i)

    results = {}

    # ----- Answerable retrieval metrics -----
    def eval_answerable(indices: List[int], name: str):
        rows = []
        for k in ks:
            mrr = 0.0
            rec = 0.0
            prec = 0.0
            n = 0

            for i in indices:
                ex = instances[i]
                if not ex.get("answerable", False):
                    continue
                gold_docs = ex.get("source_docs", [])
                if not gold_docs:
                    continue  # should not happen for answerable, but safe

                pred_docs = preds[i].get("documents", []) or []
                mrr += _reciprocal_rank(pred_docs, gold_docs, k)
                rec += _recall_at_k(pred_docs, gold_docs, k)
                prec += _precision_at_k(pred_docs, gold_docs, k)
                n += 1

            if n == 0:
                rows.append((k, None, None, None, 0))
            else:
                rows.append((k, mrr/n, rec/n, prec/n, n))

        results[name] = rows

    # all answerable
    eval_answerable(list(range(len(instances))), "answerable_all")

    # answerable by difficulty (L1/L2/L3)
    for diff in sorted(by_diff.keys()):
        eval_answerable(by_diff[diff], f"answerable_{diff}")

    # ----- Unanswerable formatting accuracy (L4/L5) -----
    def eval_unanswerable(indices: List[int], name: str):
        n_l4 = n_l5 = 0
        ok_l4 = ok_l5 = 0

        for i in indices:
            ex = instances[i]
            if ex.get("answerable", True):
                continue

            diff = ex.get("difficulty", "")
            pred_docs = preds[i].get("documents", []) or []
            # L4 expects empty docs
            if diff == "L4":
                n_l4 += 1
                if len(pred_docs) == 0:
                    ok_l4 += 1
            # L5 expects non-empty docs
            if diff == "L5":
                n_l5 += 1
                if len(pred_docs) > 0:
                    ok_l5 += 1

        results[name] = {
            "L4_doc_empty_acc": (ok_l4 / n_l4) if n_l4 else None,
            "L4_count": n_l4,
            "L5_doc_nonempty_acc": (ok_l5 / n_l5) if n_l5 else None,
            "L5_count": n_l5,
        }

    eval_unanswerable(list(range(len(instances))), "unanswerable_format_all")

    return results

def pretty_print_eval(results: Dict):
    # Print answerable tables
    for key in results:
        if key.startswith("answerable_"):
            print(f"\n=== {key} ===")
            rows = results[key]
            for (k, mrr, rec, prec, n) in rows:
                if n == 0:
                    print(f"k={k}: n=0")
                else:
                    print(f"k={k}: n={n:4d} | MRR@{k}={mrr:.4f} | Recall@{k}={rec:.4f} | Precision@{k}={prec:.4f}")

    # Print unanswerable format
    if "unanswerable_format_all" in results:
        u = results["unanswerable_format_all"]
        print("\n=== unanswerable_format_all ===")
        print(f"L4: n={u['L4_count']} | doc-empty acc={u['L4_doc_empty_acc']}")
        print(f"L5: n={u['L5_count']} | doc-nonempty acc={u['L5_doc_nonempty_acc']}")

In [13]:
# =========================
# 10) Quick Tests
# =========================

tests = [
    "How many ONE vessels had the new algorithmic routing deployed across Sharpsea Bloc shipping lanes under Phase II of the Sharpsea Route Optimization Program?",
    "What is the current FDA approval status for human somatic gene therapy treatments as of 2024?",
    "What was the total cargo and vessel damage loss ONE Network Enterprises suffered in 76 PCE, given that the Sharpsea approaches accounted for 52% of all piracy incidents?",
]

for q in tests:
    pred = answer_question(q)
    print("="*80)
    print("Q:", q)
    print("A:", pred["answer"])
    print("Docs:", pred["documents"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Q: How many ONE vessels had the new algorithmic routing deployed across Sharpsea Bloc shipping lanes under Phase II of the Sharpsea Route Optimization Program?
A: 74
Docs: ['DOC-0255', 'DOC-0161', 'DOC-0190']
Q: What is the current FDA approval status for human somatic gene therapy treatments as of 2024?
A: 
Docs: []


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What was the total cargo and vessel damage loss ONE Network Enterprises suffered in 76 PCE, given that the Sharpsea approaches accounted for 52% of all piracy incidents?
A: 
Docs: ['DOC-0174', 'DOC-0175', 'DOC-0308']


Section 11 — Dataset loop (local evaluation style)

In [14]:
# quick dev sample: 50 per difficulty

DATA_PATH = "/home/jupyter/advanced/nlp/nlp.jsonl"

instances = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            instances.append(json.loads(line))


sample = []
by_diff = defaultdict(list)
for ex in instances:
    by_diff[ex["difficulty"]].append(ex)

for diff, rows in by_diff.items():
    sample.extend(rows[:50])

preds = [answer_question(ex["question"]) for ex in sample]
results = evaluate_retrieval(sample, preds, ks=(1,3,5))
pretty_print_eval(results)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs


=== answerable_all ===
k=1: n= 150 | MRR@1=0.7667 | Recall@1=0.7667 | Precision@1=0.7667
k=3: n= 150 | MRR@3=0.8178 | Recall@3=0.8800 | Precision@3=0.3756
k=5: n= 150 | MRR@5=0.8178 | Recall@5=0.8800 | Precision@5=0.2267

=== answerable_L1 ===
k=1: n=  50 | MRR@1=0.7400 | Recall@1=0.7400 | Precision@1=0.7400
k=3: n=  50 | MRR@3=0.8133 | Recall@3=0.9200 | Precision@3=0.3067
k=5: n=  50 | MRR@5=0.8133 | Recall@5=0.9200 | Precision@5=0.1840

=== answerable_L2 ===
k=1: n=  50 | MRR@1=0.7200 | Recall@1=0.7200 | Precision@1=0.7200
k=3: n=  50 | MRR@3=0.7800 | Recall@3=0.8400 | Precision@3=0.2800
k=5: n=  50 | MRR@5=0.7800 | Recall@5=0.8400 | Precision@5=0.1680

=== answerable_L3 ===
k=1: n=  50 | MRR@1=0.8400 | Recall@1=0.8400 | Precision@1=0.8400
k=3: n=  50 | MRR@3=0.8600 | Recall@3=0.8800 | Precision@3=0.5400
k=5: n=  50 | MRR@5=0.8600 | Recall@5=0.8800 | Precision@5=0.3280

=== answerable_L4 ===
k=1: n=0
k=3: n=0
k=5: n=0

=== answerable_L5 ===
k=1: n=0
k=3: n=0
k=5: n=0

=== unanswerab

In [15]:
# =========================
# 11) Run over nlp.jsonl (local)
# =========================

DATA_PATH = "/home/jupyter/advanced/nlp/nlp.jsonl"

instances = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            instances.append(json.loads(line))

preds = []
for ex in instances:
    preds.append(answer_question(ex["question"]))

results = evaluate_retrieval(instances, preds, ks=(1,3,5))
pretty_print_eval(results)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs


=== answerable_all ===
k=1: n= 943 | MRR@1=0.8335 | Recall@1=0.8335 | Precision@1=0.8335
k=3: n= 943 | MRR@3=0.8823 | Recall@3=0.9406 | Precision@3=0.3291
k=5: n= 943 | MRR@5=0.8832 | Recall@5=0.9449 | Precision@5=0.1985

=== answerable_L1 ===
k=1: n= 592 | MRR@1=0.8598 | Recall@1=0.8598 | Precision@1=0.8598
k=3: n= 592 | MRR@3=0.9077 | Recall@3=0.9662 | Precision@3=0.3221
k=5: n= 592 | MRR@5=0.9087 | Recall@5=0.9713 | Precision@5=0.1943

=== answerable_L2 ===
k=1: n= 291 | MRR@1=0.7766 | Recall@1=0.7766 | Precision@1=0.7766
k=3: n= 291 | MRR@3=0.8322 | Recall@3=0.8969 | Precision@3=0.2990
k=5: n= 291 | MRR@5=0.8330 | Recall@5=0.9003 | Precision@5=0.1801

=== answerable_L3 ===
k=1: n=  60 | MRR@1=0.8500 | Recall@1=0.8500 | Precision@1=0.8500
k=3: n=  60 | MRR@3=0.8750 | Recall@3=0.9000 | Precision@3=0.5444
k=5: n=  60 | MRR@5=0.8750 | Recall@5=0.9000 | Precision@5=0.3300

=== answerable_L4 ===
k=1: n=0
k=3: n=0
k=5: n=0

=== answerable_L5 ===
k=1: n=0
k=3: n=0
k=5: n=0

=== unanswerab

Evaluation - retrivel

In [20]:
# Find L4 failures (pred returned docs when it should not)
fail_l5 = []
for ex, pr in zip(instances, preds):
    if ex.get("difficulty") == "L5" and not ex.get("answerable", True):
        if (pr.get("documents") or []):
            fail_l5.append((ex["key"], ex["question"], pr["documents"]))

print("L5 failures:", len(fail_l5))
for key, q, docs in fail_l5[:20]:
    print("\nKEY:", key)
    print("Q:", q)
    print("Pred docs:", docs)

L5 failures: 33

KEY: 82
Q: What was the total cargo and vessel damage loss ONE Network Enterprises suffered in 76 PCE, given that the Sharpsea approaches accounted for 52% of all piracy incidents?
Pred docs: ['DOC-0174', 'DOC-0175', 'DOC-0308']

KEY: 128
Q: What was the net user growth figure for Edge nanobots in Haven during 76 PCE if the 3.1 million Opting losses are subtracted from new activations?
Pred docs: ['DOC-0319', 'DOC-0126']

KEY: 137
Q: Which regulatory body other than TEC Environmental Compliance classified the Terminal 6 nanobot breach as Severity Level 3?
Pred docs: ['DOC-0136', 'DOC-0066', 'DOC-0328']

KEY: 205
Q: What motivated Phyrexis Group to publicly claim credit for funding the Edge launch event in Haven?
Pred docs: ['DOC-0274', 'DOC-0235']

KEY: 212
Q: Which article of the Monteverde Convention does the CGC cite as establishing its jurisdiction over maritime approaches to Haven Island?
Pred docs: ['DOC-0051', 'DOC-0216', 'DOC-0257']

KEY: 233
Q: Which two Direc

In [22]:
# Find L4 failures (pred returned docs when it should not)
fail_l1 = []
for ex, pr in zip(instances, preds):
    if ex.get("difficulty") == "L1" and not ex.get("answerable", True):
        if (pr.get("documents") or []):
            fail_l4.append((ex["key"], ex["question"], pr["documents"]))

print("L1 failures:", len(fail_l1))
for key, q, docs in fail_l1[:20]:
    print("\nKEY:", key)
    print("Q:", q)
    print("Pred docs:", docs)

L1 failures: 0


In [23]:
# Find L2 failures (pred returned docs when it should not)
fail_l2 = []
for ex, pr in zip(instances, preds):
    if ex.get("difficulty") == "L2" and not ex.get("answerable", True):
        if (pr.get("documents") or []):
            fail_l4.append((ex["key"], ex["question"], pr["documents"]))

print("L2 failures:", len(fail_l2))
for key, q, docs in fail_l2[:20]:
    print("\nKEY:", key)
    print("Q:", q)
    print("Pred docs:", docs)

L2 failures: 0


In [24]:
# Find L4 failures (pred returned docs when it should not)
fail_l3 = []
for ex, pr in zip(instances, preds):
    if ex.get("difficulty") == "L3" and not ex.get("answerable", True):
        if (pr.get("documents") or []):
            fail_l3.append((ex["key"], ex["question"], pr["documents"]))

print("L3 failures:", len(fail_l3))
for key, q, docs in fail_l3[:20]:
    print("\nKEY:", key)
    print("Q:", q)
    print("Pred docs:", docs)

L3 failures: 0


In [17]:
import time

# =========================================
# SIMPLE LOCAL EVALUATION
# =========================================

test_questions = [
    "Who controls society in Haven?",
    "Who governs Central Haven?",
    "What are the Floodwalls?",
    "What is Opting?",
    "Who maintains the Floodwalls?"
]

results = []

start_total = time.time()

for q in test_questions:

    start = time.time()

    result = answer_question(q)

    end = time.time()

    runtime = end - start

    results.append(runtime)

    print("=" * 80)
    print("QUESTION:")
    print(q)

    print("\nANSWER:")
    print(result["answer"])

    print("\nDOCUMENTS:")
    print(result["documents"])

    print("\nTIME:")
    print(round(runtime, 2), "seconds")

end_total = time.time()

# =========================================
# SPEED METRICS
# =========================================

avg_time = sum(results) / len(results)

total_time = end_total - start_total

# competition speed formula
# assuming t_max = 30 seconds

speed_score = 1 - (min(avg_time, 30) / 30)

print("\n" + "=" * 80)
print("FINAL METRICS")
print("=" * 80)

print("Average Time Per Question:", round(avg_time, 2), "sec")

print("Total Runtime:", round(total_time, 2), "sec")

print("Estimated Speed Score:", round(speed_score, 3))

# =========================================
# MANUAL ACCURACY ESTIMATE
# =========================================

print("\nMANUAL ACCURACY CHECK")
print("Count how many answers are correct.")
print("Example: 4 correct out of 5 = 0.80 accuracy")

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Who controls society in Haven?

ANSWER:
The CGC Civic Administration controls society in Haven through its administration of North Haven and Central Haven. The passage states that "No standing mechanism exists to integrate these entities into a single operational authority during emergencies declared at or near sector boundaries." This indicates that while there is no central governing body overseeing the entire island, each sector has

DOCUMENTS:
['DOC-0163', 'DOC-0083', 'DOC-0015', 'DOC-0314', 'DOC-0007']

TIME:
3.04 seconds


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Who governs Central Haven?

ANSWER:
The CGC governs Central Haven directly as a body.

DOCUMENTS:
['DOC-0083', 'DOC-0163', 'DOC-0340']

TIME:
0.87 seconds


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What are the Floodwalls?

ANSWER:
The Floodwalls are Cyanite's single most publicly prominent asset and the engineering infrastructure that keeps Haven's 100 million residents from an extremely wet fate. They consist of a nested system of walls, channels, retaining tanks, and nuclear-fission-powered pumps that keep the below-sea-level city dry. Notably

DOCUMENTS:
['DOC-0228', 'DOC-0086', 'DOC-0202']

TIME:
2.88 seconds


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What is Opting?

ANSWER:
Purging Edge nanobots and their influence from one's body.

DOCUMENTS:
['DOC-0299', 'DOC-0068', 'DOC-0306', 'DOC-0128']

TIME:
1.02 seconds


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Who maintains the Floodwalls?

ANSWER:
Cyanite Industries

DOCUMENTS:
['DOC-0089', 'DOC-0096', 'DOC-0189', 'DOC-0086']

TIME:
0.6 seconds

FINAL METRICS
Average Time Per Question: 1.68 sec
Total Runtime: 8.41 sec
Estimated Speed Score: 0.944

MANUAL ACCURACY CHECK
Count how many answers are correct.
Example: 4 correct out of 5 = 0.80 accuracy
